In [ ]:
"""
Map Detection Training Notebook

This notebook trains a YOLOv8 model to detect maps in images.
The pipeline includes dataset preparation, train/val splitting, model training,
validation, and model export.

Author: NCSU - Fall 2025 Board Game Project
"""

# ============================================================================
# IMPORTS
# ============================================================================

import os
import shutil
import random
from pathlib import Path
from dotenv import load_dotenv
from ultralytics import YOLO  # YOLOv8 implementation
import yaml  # For creating configuration files
import torch  # PyTorch deep learning framework
from torch.serialization import add_safe_globals
from ultralytics.nn.tasks import DetectionModel

# ============================================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================================

# Load .env from the map_detection/ directory (one level up from src/)
load_dotenv(dotenv_path=Path(__file__).resolve().parent.parent / ".env")

DATASET_DIR      = Path(os.environ["DATASET_DIR"])
PRETRAINED_MODEL = os.environ["PRETRAINED_MODEL"]
MODEL_SAVE_DIR   = Path(os.environ["MODEL_SAVE_DIR"])
MODEL_SAVE_NAME  = os.environ["MODEL_SAVE_NAME"]


In [ ]:
# ============================================================================
# STEP 1: SETUP DATASET PATHS
# ============================================================================

# Paths to images and labels (set via .env)
images_dir = DATASET_DIR / "images"
labels_dir = DATASET_DIR / "labels"

# Output directory structure for YOLO training
output_dir   = DATASET_DIR / "yolo_dataset"
train_images = output_dir / "train/images"
train_labels = output_dir / "train/labels"
val_images   = output_dir / "val/images"
val_labels   = output_dir / "val/labels"

# Create output directories
for d in [train_images, train_labels, val_images, val_labels]:
    d.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================================
# STEP 2: TRAIN/VALIDATION SPLIT
# ============================================================================

# Collect all image files from augmented dataset
all_images = list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png"))

# Set random seed for reproducible splits
random.seed(42)
random.shuffle(all_images)

# Split dataset: 80% training, 20% validation
split_idx = int(0.8 * len(all_images))
train_files = all_images[:split_idx]
val_files   = all_images[split_idx:]

# Function to copy image and label files to train/val directories
def copy_files(file_list, dest_img_dir, dest_lbl_dir):
    """Copy images and their corresponding label files to destination"""
    for img_path in file_list:
        # Find corresponding label file
        lbl_path = labels_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            # Copy both image and label
            shutil.copy(img_path, dest_img_dir / img_path.name)
            shutil.copy(lbl_path, dest_lbl_dir / lbl_path.name)

# Copy files to their respective directories (uncomment to run)
copy_files(train_files, train_images, train_labels)
copy_files(val_files, val_images, val_labels)

print(f"Copied {len(train_files)} train and {len(val_files)} val images.")

In [ ]:
# ============================================================================
# STEP 3: CREATE YOLO CONFIGURATION FILE (data.yaml)
# ============================================================================

# YOLO requires a data.yaml file that specifies:
# - Paths to training and validation datasets
# - Number of classes (nc)
# - Class names

data_yaml = {
    "train": str(train_images.parent.resolve()),  # Path to train directory
    "val": str(val_images.parent.resolve()),      # Path to val directory
    "nc": 1,                                      # Number of classes (1 = map)
    "names": ["map"]                            # Class name(s)
}

# Write configuration to data.yaml file (uncomment to run)
with open(output_dir / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"data.yaml created at {output_dir / 'data.yaml'}")

In [ ]:
# ============================================================================
# STEP 4: CHECK FOR GPU AVAILABILITY
# ============================================================================

# Check if CUDA-compatible GPU is available
# Training on GPU is significantly faster than CPU
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"  # Fall back to CPU
    print("⚠️ CUDA not available, falling back to CPU")
    print("Note: Training on CPU will be much slower. Consider using Google Colab with GPU.")

In [ ]:
# ============================================================================
# STEP 5: TRAIN YOLO MODEL
# ============================================================================

# ---- Hyperparameters (tune these directly here) ----
EPOCHS     = 20
IMG_SIZE   = 640
BATCH_SIZE = 4
DEVICE     = device
RUN_NAME   = "map-detector"

# Load pre-trained YOLO model (weight file set via .env)
model = YOLO(PRETRAINED_MODEL)

# Train the model
results = model.train(
    data=str(output_dir / "data.yaml"),  # Path to data configuration file
    epochs=EPOCHS,                       # Number of training epochs
    imgsz=IMG_SIZE,                      # Input image size
    batch=BATCH_SIZE,                    # Batch size (reduce if out of memory)
    device=DEVICE,                       # Device: "cpu" or 0 for first GPU
    workers=0,                           # 0 = disable multiprocessing
    cache=True,                          # Cache images in RAM for faster training
    patience=5,                          # Early stopping patience
    name=RUN_NAME,                       # Name for this training run
    exist_ok=True,                       # Allow overwriting existing run
    verbose=True                         # Print training progress
)

# Training outputs:
# - Model checkpoints: runs/detect/<RUN_NAME>/weights/
# - Best model: best.pt, Last model: last.pt


In [ ]:
# ============================================================================
# STEP 6: VALIDATE MODEL PERFORMANCE
# ============================================================================

# Run validation on the validation set
# This calculates metrics like precision, recall, mAP (mean Average Precision)
metrics = model.val()

# Print validation metrics
print("\n=== Validation Metrics ===")
print(metrics)
print("\nKey metrics to check:")
print("- mAP50: Mean Average Precision at IoU threshold 0.5")
print("- mAP50-95: Mean Average Precision averaged over IoU 0.5-0.95")
print("- Precision: Accuracy of positive predictions")
print("- Recall: Fraction of actual maps detected")

In [ ]:
# ============================================================================
# STEP 7: TEST INFERENCE ON SAMPLE IMAGE
# ============================================================================

# Run inference on a random validation image to test the model
test_img = random.choice(val_files)
pred = model(str(test_img), save=True)

print(f"Inference saved for {test_img.name}")
print(f"Results saved to: runs/detect/predict/")
print("\nTo view the result, check the generated image with bounding boxes.")

In [ ]:
# ============================================================================
# STEP 8: SAVE TRAINED MODEL
# ============================================================================

# Create directory for saving models (path from .env)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy the best weights from training to saved_models directory
model_path = MODEL_SAVE_DIR / MODEL_SAVE_NAME
shutil.copy(f"runs/detect/{RUN_NAME}/weights/best.pt", model_path)
print(f"✅ Model saved to {model_path}")

# ============================================================================
# LOADING AND USING THE SAVED MODEL
# ============================================================================

print(val_files)
# Load the saved model for inference
loaded_model = YOLO(model_path)

def detect_map(image_path):
    """
    Detect maps in an image using the trained model.

    Args:
        image_path (str): Path to the image file

    Returns:
        Detection results containing bounding boxes, confidence scores, etc.
    """
    results = loaded_model(image_path)
    return results[0]  # Return first (and only) detection result

# Test the loaded model on a validation image
test_img = random.choice(val_files)
result = detect_map(str(test_img))
print(f"\n=== Detection Test ===")
print(f"Test image: {test_img.name}")
print(f"Detected {len(result.boxes)} maps in test image")

# Access detection details
if len(result.boxes) > 0:
    for i, box in enumerate(result.boxes):
        print(f"\nMap {i+1}:")
        print(f"  - Confidence: {box.conf[0]:.3f}")
        print(f"  - Bounding box: {box.xyxy[0].tolist()}")  # [x1, y1, x2, y2]


In [ ]:
# ============================================================================ 
# STEP 9: FIND IMAGES WITH INACCURATE MAP DETECTION
# ============================================================================ 

import numpy as np
from PIL import Image

# ---- Tunable threshold (adjust here directly) ----
IOU_THRESHOLD = 0.5

def compute_iou(boxA, boxB):
    # boxA and boxB: [x1, y1, x2, y2]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-6)
    return iou

def parse_yolo_label(label_path, img_w, img_h):
    # YOLO format: class x_center y_center width height (all normalized)
    with open(label_path, 'r') as f:
        line = f.readline().strip()
        if not line:
            return None
        parts = line.split()
        if len(parts) < 5:
            return None
        _, x, y, w, h = map(float, parts)
        x1 = (x - w/2) * img_w
        y1 = (y - h/2) * img_h
        x2 = (x + w/2) * img_w
        y2 = (y + h/2) * img_h
        return [x1, y1, x2, y2]

bad_images = []

for img_path in val_files:
    label_path = labels_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        continue
    img = Image.open(img_path)
    img_w, img_h = img.size
    gt_box = parse_yolo_label(label_path, img_w, img_h)
    if gt_box is None:
        continue
    result = detect_map(str(img_path))
    pred_boxes = [box.xyxy[0].cpu().numpy() for box in result.boxes]
    if not pred_boxes:
        bad_images.append(str(img_path))
        continue
    ious = [compute_iou(gt_box, pred_box) for pred_box in pred_boxes]
    max_iou = max(ious) if ious else 0
    if max_iou < IOU_THRESHOLD:
        bad_images.append(str(img_path))

print(f"Images with inaccurate map detection (IoU < {IOU_THRESHOLD}):")
for bad_img in bad_images:
    print(bad_img)


In [ ]:
# ============================================================================ 
# STEP 10: VISUALIZE PREDICTED VS GROUND TRUTH BOXES
# ============================================================================ 

import matplotlib.pyplot as plt
import matplotlib.patches as patches

def show_image_with_boxes(img_path, gt_box, pred_boxes, iou_list=None):
    img = Image.open(img_path)
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img)
    # Draw ground truth box (green)
    if gt_box is not None:
        rect = patches.Rectangle((gt_box[0], gt_box[1]), gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                                 linewidth=2, edgecolor='g', facecolor='none', label='Ground Truth')
        ax.add_patch(rect)
    # Draw predicted boxes (red)
    for i, pred_box in enumerate(pred_boxes):
        rect = patches.Rectangle((pred_box[0], pred_box[1]), pred_box[2]-pred_box[0], pred_box[3]-pred_box[1],
                                 linewidth=2, edgecolor='r', facecolor='none', label='Prediction' if i==0 else None)
        ax.add_patch(rect)
        if iou_list is not None:
            ax.text(pred_box[0], pred_box[1]-10, f'IoU: {iou_list[i]:.2f}', color='red', fontsize=10)
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys())
    plt.axis('off')
    plt.show()

# Example: Visualize a bad image (first in the list)
if bad_images:
    img_path = bad_images[-1]
    label_path = labels_dir / (Path(img_path).stem + ".txt")
    img = Image.open(img_path)
    img_w, img_h = img.size
    gt_box = parse_yolo_label(label_path, img_w, img_h)
    result = detect_map(str(img_path))
    pred_boxes = [box.xyxy[0].cpu().numpy() for box in result.boxes]
    ious = [compute_iou(gt_box, pred_box) for pred_box in pred_boxes] if gt_box is not None and pred_boxes else None
    show_image_with_boxes(img_path, gt_box, pred_boxes, ious)

# To visualize a good image, pick one not in bad_images
good_images = [str(p) for p in val_files if str(p) not in bad_images]
if good_images:
    img_path = good_images[-1]
    label_path = labels_dir / (Path(img_path).stem + ".txt")
    img = Image.open(img_path)
    img_w, img_h = img.size
    gt_box = parse_yolo_label(label_path, img_w, img_h)
    result = detect_map(str(img_path))
    pred_boxes = [box.xyxy[0].cpu().numpy() for box in result.boxes]
    ious = [compute_iou(gt_box, pred_box) for pred_box in pred_boxes] if gt_box is not None and pred_boxes else None
    show_image_with_boxes(img_path, gt_box, pred_boxes, ious)